In [ ]:
import pandas as pd
import numpy as np

# === Load edge change data ===
df_edge = pd.read_csv(
    r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Forest_Edge_Change_historical_Disturbance_Attribution_1988_2021_pa&ownership_reprojected.csv"
)

# === Load region edge length data from CSV and skip first year ===
region_edge_df = pd.read_csv(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\forest_edge_by_region.csv")
region_edge_df = region_edge_df[region_edge_df['Year'] != 1988]
region_edge_df.set_index('Year', inplace=True)

# === Assign region based on Ecoregion ===
state_to_region_division = {
    'Connecticut': ('Northeast', 'New England'), 'Maine': ('Northeast', 'New England'),
    'Massachusetts': ('Northeast', 'New England'), 'New Hampshire': ('Northeast', 'New England'),
    'Rhode Island': ('Northeast', 'New England'), 'Vermont': ('Northeast', 'New England'),
    'New Jersey': ('Northeast', 'Middle Atlantic'), 'New York': ('Northeast', 'Middle Atlantic'),
    'Pennsylvania': ('Northeast', 'Middle Atlantic'),
    'Illinois': ('Midwest', 'East North Central'), 'Indiana': ('Midwest', 'East North Central'),
    'Michigan': ('Midwest', 'East North Central'), 'Ohio': ('Midwest', 'East North Central'),
    'Wisconsin': ('Midwest', 'East North Central'),
    'Iowa': ('Midwest', 'West North Central'), 'Kansas': ('Midwest', 'West North Central'),
    'Minnesota': ('Midwest', 'West North Central'), 'Missouri': ('Midwest', 'West North Central'),
    'Nebraska': ('Midwest', 'West North Central'), 'North Dakota': ('Midwest', 'West North Central'),
    'South Dakota': ('Midwest', 'West North Central'),
    'Delaware': ('South', 'South Atlantic'), 'District of Columbia': ('South', 'South Atlantic'),
    'Florida': ('South', 'South Atlantic'), 'Georgia': ('South', 'South Atlantic'),
    'Maryland': ('South', 'South Atlantic'), 'North Carolina': ('South', 'South Atlantic'),
    'South Carolina': ('South', 'South Atlantic'), 'Virginia': ('South', 'South Atlantic'),
    'West Virginia': ('South', 'South Atlantic'),
    'Alabama': ('South', 'East South Central'), 'Kentucky': ('South', 'East South Central'),
    'Mississippi': ('South', 'East South Central'), 'Tennessee': ('South', 'East South Central'),
    'Arkansas': ('South', 'West South Central'), 'Louisiana': ('South', 'West South Central'),
    'Oklahoma': ('South', 'West South Central'), 'Texas': ('South', 'West South Central'),
    'Arizona': ('West', 'Mountain'), 'Colorado': ('West', 'Mountain'), 'Idaho': ('West', 'Mountain'),
    'Montana': ('West', 'Mountain'), 'Nevada': ('West', 'Mountain'), 'New Mexico': ('West', 'Mountain'),
    'Utah': ('West', 'Mountain'), 'Wyoming': ('West', 'Mountain'),
    'Alaska': ('West', 'Pacific'), 'California': ('West', 'Pacific'),
    'Hawaii': ('West', 'Pacific'), 'Oregon': ('West', 'Pacific'), 'Washington': ('West', 'Pacific')
}
df_edge['Region'] = df_edge['Ecoregion'].map(lambda x: state_to_region_division.get(x, ('Unknown',))[0])

# === Adjust Year and calculate signed edge length ===
df_edge['Year'] += 1
edge_sign_mapping = {1: -1, 2: -1, 3: 1, 4: 1, 5: 0}
df_edge['Signed_EdgeLength_km'] = df_edge['PixelCount'] * 30 / 1000 * df_edge['EdgeDynamic'].map(edge_sign_mapping)
df_edge.loc[df_edge['DisturbanceCategory'] == 0, 'GapYears'] = 0

# === Filter stable vs dynamic ===
stable_df = df_edge[df_edge['EdgeDynamic'] == 5].copy()
net_df = df_edge[df_edge['EdgeDynamic'] != 5].copy()

# === Total edge length from provided CSV ===
region_total_edge_length = region_edge_df[['South (km)', 'West (km)', 'Northeast (km)', 'Midwest (km)']]
region_total_edge_length.columns = ['South', 'West', 'Northeast', 'Midwest']

# === Net edge change by disturbance category (merge 0 -> 8) ===
region_edge_change = {}
for region in ['Northeast', 'Midwest', 'South', 'West']:
    df_region = net_df[net_df['Region'] == region]
    grouped = (
        df_region.groupby(['Year', 'DisturbanceCategory'])['Signed_EdgeLength_km']
        .sum().reset_index()
    )
    pivoted = grouped.pivot(index='Year', columns='DisturbanceCategory',
                            values='Signed_EdgeLength_km').fillna(0)

    # ensure all 0..8 exist
    for cat in range(9):
        if cat not in pivoted.columns:
            pivoted[cat] = 0

    # MERGE "No Disturbance Detected" (0) INTO "Other" (8)
    pivoted[8] = pivoted[8] + pivoted[0]
    pivoted = pivoted.drop(columns=[0])

    # keep columns ordered 1..8
    pivoted = pivoted[[1, 2, 3, 4, 5, 6, 7, 8]]

    region_edge_change[region] = pivoted

In [ ]:
import matplotlib.pyplot as plt

# === Plot ===
fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharex=True, dpi=600)
axes = axes.flatten()

dist_label = {
    1: 'Logging', 2: 'Construction', 3: 'Stress',
    4: 'Natural Hazard', 5: 'Water Dynamic', 6: 'Fire',
    7: 'Agriculture Activity', 8: 'Other'
}
disturbance_colors_edge = {
    'Logging': '#1b9e77', 'Construction': 'purple', 'Stress': '#e7298a',
    'Natural Hazard': '#66a61e', 'Water Dynamic': '#1f78b4',
    'Fire': 'red', 'Agriculture Activity': 'gold', 'Other': 'gray'
}

for idx, region in enumerate(['Northeast', 'Midwest', 'South', 'West']):
    ax = axes[idx]
    pivoted = region_edge_change[region]
    all_years = sorted(set(pivoted.index).union(set(region_total_edge_length[region].index)))
    pivoted = pivoted.reindex(all_years, fill_value=0)
    total_edge_length = region_total_edge_length[region].reindex(all_years, fill_value=np.nan)

    bar_x = np.arange(len(all_years))
    bar_colors = [disturbance_colors_edge[dist_label[cat]] for cat in pivoted.columns]

    pivoted.plot(kind='bar', stacked=True, ax=ax, color=bar_colors, width=0.9, legend=False)
    ax.axhline(0, color='gray', linestyle=':')

    ax.set_ylabel("Forest Edge Length Change (km)")
    ax.set_title(f"({chr(97+idx)}) {region}", fontsize=12, fontweight='bold', loc='left')
    ax.set_xticks(bar_x)
    ax.set_xticklabels(all_years, rotation=45)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(5, 5))

    # Right axis: total edge length
    ax_right = ax.twinx()
    ax_right.plot(bar_x, total_edge_length.values, color='green', linewidth=2, linestyle='-', marker='x')
    ax_right.set_ylabel("Total Forest Edge Length (km)", color='green')
    ax_right.tick_params(axis='y', colors='green')
    ax_right.ticklabel_format(axis='y', style='sci', scilimits=(7, 7))

# Add legend to last axis
cats = [1,2,3,4,5,6,7,8]
handles = [plt.Rectangle((0, 0), 1, 1, color=disturbance_colors_edge[dist_label[c]]) for c in cats]
axes[-1].legend(handles, [dist_label[c] for c in cats], bbox_to_anchor=(0.0, 0.99), loc='upper left')

plt.tight_layout()
plt.savefig(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Figure_S4.png", dpi=600, bbox_inches='tight')
plt.show()